[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/Multimodal-Deep-Learning/blob/main/07_traing_problem_and%20soultion/05_training_instability/05_training_instability.ipynb)

# 05. Training Instability, NaN/Inf & Convergence

---


In [ ]:
# ============================================================
#  Colab Setup (run this cell first if on Google Colab)
# ============================================================
import os, sys

if 'google.colab' in sys.modules:
    !git clone https://github.com/Gaurav14cs17/Multimodal-Deep-Learning.git
    os.chdir('Multimodal-Deep-Learning')
    os.chdir('07_traing_problem_and soultion/05_training_instability')
    !pip install -q torch torchvision matplotlib numpy
else:
    nb_dir = os.getcwd()
    if not os.path.basename(nb_dir) == '05_training_instability':
        os.chdir(os.path.join(os.path.dirname(os.path.abspath('__file__')), '..', '..', '05_training_instability'))
    sys.path.append(os.path.join(os.getcwd(), '..', '..'))

print(f'Working directory: {os.getcwd()}')


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)
plt.rcParams.update({'figure.figsize': (10, 5), 'font.size': 11})


## 0. How NaN is Created — IEEE 754 Arithmetic Rules

Every floating-point NaN in PyTorch comes from one of **seven IEEE 754 operations** (or propagation from an existing NaN):

| # | Operation | Result |
|---|-----------|--------|
| 1 | $0 / 0$ | NaN |
| 2 | $\infty - \infty$ | NaN |
| 3 | $\infty + (-\infty)$ | NaN |
| 4 | $0 \times \infty$ | NaN |
| 5 | $\infty / \infty$ | NaN |
| 6 | $\sqrt{\text{negative}}$ | NaN |
| 7 | Any op involving existing NaN | NaN (propagation) |

### Bit-Level Representation

**FP32:** sign (1 bit) + exponent (8 bits) + mantissa (23 bits)

- **NaN:** all exponent bits = 1, mantissa $\neq 0$
- **Inf:** all exponent bits = 1, mantissa = 0
- **Normal:** exponent $\in [1, 254]$

**FP16:** sign (1) + exponent (5) + mantissa (10) → **max = 65504**

**BF16:** sign (1) + exponent (8) + mantissa (7) → **max $\approx 3.39 \times 10^{38}$** (same exponent range as FP32!)

This is why BF16 rarely overflows where FP16 dies: same dynamic range, fewer mantissa bits.


In [ ]:
# Demonstrate all 7 IEEE 754 NaN creation rules
rules = [
    ("0/0", lambda: torch.tensor(0.) / 0),
    ("inf - inf", lambda: torch.tensor(float('inf')) - torch.tensor(float('inf'))),
    ("inf + (-inf)", lambda: torch.tensor(float('inf')) + torch.tensor(float('-inf'))),
    ("0 * inf", lambda: torch.tensor(0.) * torch.tensor(float('inf'))),
    ("inf / inf", lambda: torch.tensor(float('inf')) / torch.tensor(float('inf'))),
    ("sqrt(-1)", lambda: torch.sqrt(torch.tensor(-1.))),
    ("NaN propagation", lambda: torch.tensor(float('nan')) + 1),
]
print("IEEE 754 NaN creation rules:")
for name, fn in rules:
    result = fn().item()
    print(f"  {name:20s} = {result}")


In [ ]:
# FP16 vs BF16 vs FP32 — range, epsilon, subnormals
dtypes = [torch.float32, torch.float16, torch.bfloat16]
rows = []
for dt in dtypes:
    fi = torch.finfo(dt)
    rows.append({
        'dtype': str(dt).split('.')[-1],
        'max': fi.max,
        'min': fi.min,
        'smallest_normal': fi.tiny,
        'eps': fi.eps,
        'bits': {torch.float32: '1+8+23', torch.float16: '1+5+10', torch.bfloat16: '1+8+7'}[dt],
    })

print(f"{'dtype':<10} {'bits':<8} {'max':>12} {'min_normal':>12} {'eps':>12}")
print("-" * 58)
for r in rows:
    print(f"{r['dtype']:<10} {r['bits']:<8} {r['max']:>12.4e} {r['smallest_normal']:>12.4e} {r['eps']:>12.4e}")

# Visual: representable range comparison
fig, ax = plt.subplots(figsize=(10, 4))
labels = [r['dtype'] for r in rows]
maxs = [r['max'] for r in rows]
colors = ['#2196F3', '#FF5722', '#4CAF50']
bars = ax.bar(labels, maxs, color=colors)
ax.set_yscale('log')
ax.set_ylabel('Max representable value')
ax.set_title('FP32 vs FP16 vs BF16 — Dynamic Range (log scale)')
for bar, val in zip(bars, maxs):
    ax.text(bar.get_x() + bar.get_width()/2, val * 1.2, f'{val:.2e}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

print("\nKey insight: FP16 max=65504 — any activation/logit above this overflows to Inf → NaN.")
print("BF16 shares FP32 exponent range — preferred for transformer/contrastive training on A100/H100.")


## 0b. NaN is Contagious — The Propagation Problem

Once NaN enters **any** tensor, it spreads irreversibly:

```
Forward pass:  NaN input → NaN output (through EVERY subsequent layer)
Backward pass: NaN gradient → NaN weight update → ALL future outputs NaN
Distributed:   NaN on ONE GPU → AllReduce average → NaN on ALL GPUs
```

**The ONLY way to recover:** detect NaN **before** it propagates, then skip the step or fix the source.

There is no "NaN recovery" mid-training — you must restart from the last good checkpoint after fixing the root cause.


## 0c. `torch.autograd.detect_anomaly()` — The #1 Debugging Tool

**THE FIRST THING TO TRY when loss goes NaN.**

What it does:
- Wraps every backward op with a NaN/Inf check
- Prints a **full traceback** showing exactly which operation produced NaN
- Performance cost: ~**2× slower** (use only for debugging, not production)

How to read the error:
```
RuntimeError: Function 'DivBackward0' returned nan values in its 0th output.
```
→ Division produced NaN — check for division by zero in forward pass.

Persistent mode: `torch.autograd.set_detect_anomaly(True)` for entire training loop (very slow).


In [ ]:
# detect_anomaly demo — pinpoints EXACT op that produces NaN
class NaNModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.lin = nn.Linear(4, 4)
        self.lin.weight.data.fill_(0.0)  # zero weights → zero output

    def forward(self, x):
        out = self.lin(x)  # all zeros when weights are zero
        # Bad: divide by norm without epsilon → 0/0 = NaN in backward
        return out / out.norm(dim=-1, keepdim=True)

model = NaNModel()
x = torch.randn(2, 4, requires_grad=True)
target = torch.randn(2, 4)

print("Without detect_anomaly (NaN silently propagates):")
out = model(x)
loss = (out - target).pow(2).mean()
loss.backward()
print(f"  loss={loss.item()}, grad has NaN: {torch.isnan(x.grad).any().item()}")

print("\nWith detect_anomaly (shows exact backward op):")
model.zero_grad()
x.grad = None
try:
    with torch.autograd.detect_anomaly():
        out = model(x)
        loss = (out - target).pow(2).mean()
        loss.backward()
except RuntimeError as e:
    print(f"  Caught: {str(e)[:120]}...")

# Persistent mode (comment out in production — very slow)
# torch.autograd.set_detect_anomaly(True)
print("\nTip: combine detect_anomaly with forward hooks (NaNTracer below) for full picture.")


---

## 0d. Every Common NaN Source in Deep Learning (15 Cases)

Each subsection: **WHY** (math) → **HOW to detect** → **HOW to fix** → runnable demo.

Use this as a lookup table when `detect_anomaly()` points to a suspicious op.


### Source 1: `log(0)` → $-\infty$ → NaN in Loss

**WHY:** Cross-entropy internally computes $\log(\text{softmax}(x))$. When softmax output $\approx 0$ (confident wrong prediction): $\log(0) = -\infty$. Combining $0 \times (-\infty) = \text{NaN}$ in weighted losses.

**Detect:** NaN in loss backward; `detect_anomaly` points to `LogBackward`.

**Fix:** Use `F.log_softmax` + `F.nll_loss`, or clamp probs with epsilon.


In [ ]:
# Source 1: log(0) in cross-entropy path
probs = torch.tensor([0.0, 1.0])
raw_log = -torch.log(probs[0])  # inf
print(f"log(0): {raw_log.item()}  (inf → NaN when multiplied by 0*inf elsewhere)")

# Fix 1: epsilon clamp
safe_log = torch.log(probs + 1e-8)
print(f"log(prob + eps): {safe_log[0].item():.4f}")

# Fix 2: log_softmax (numerically stable — PyTorch uses log-sum-exp internally)
logits = torch.tensor([10.0, -5.0])  # confident wrong class
stable = F.log_softmax(logits, dim=0)
loss = F.nll_loss(stable.unsqueeze(0), torch.tensor([0]))
print(f"log_softmax + nll_loss: {loss.item():.4f} (finite)")


### Source 2: KL Divergence NaN

**WHY:** $\text{KL}(p \| q) = \sum_i p_i \log(p_i / q_i)$. When $q_i \approx 0$: $\log(p/q) \to \infty$. When $p_i = 0$: $0 \cdot \log(0) = 0 \cdot (-\infty) = \text{NaN}$ (IEEE 754 rule #4 variant).

**Detect:** NaN in distillation/VAE losses.

**Fix:** `reduction='batchmean'`, clamp $q \geq \epsilon$, use `F.kl_div` with log-prob input.


In [ ]:
# Source 2: KL divergence NaN
p = torch.tensor([0.5, 0.5, 0.0])
q = torch.tensor([0.3, 0.3, 0.0001])

# Bad: q nearly zero on third class, p=0 on third → NaN
kl_bad = (p * (p / q).log()).sum()
print(f"Naive KL: {kl_bad.item()}  (NaN={torch.isnan(kl_bad).item()})")

# Fix: clamp q, use F.kl_div
q_safe = q.clamp(min=1e-8)
kl_good = F.kl_div(q_safe.log(), p, reduction='batchmean')
print(f"F.kl_div (clamped): {kl_good.item():.4f}")


### Source 3: Cosine Similarity with Zero Vectors

**WHY:** $\cos(a, b) = \frac{a \cdot b}{\|a\| \|b\|}$. When $\|a\| = 0$: division by zero → NaN. Common in contrastive learning when an encoder outputs all zeros (dead ReLU, collapsed representation).

**Detect:** NaN in CLIP/SimCLR similarity matrix.

**Fix:** Pass `eps=1e-8` to `F.cosine_similarity`, or check for zero-norm vectors before normalization.


In [ ]:
# Source 3: cosine similarity with zero vector
a = torch.zeros(5)
b = torch.randn(5)
cos_bad = F.cosine_similarity(a.unsqueeze(0), b.unsqueeze(0))
print(f"Zero vector cosine: {cos_bad.item()}  (NaN={torch.isnan(cos_bad).item()})")

cos_safe = F.cosine_similarity(a.unsqueeze(0), b.unsqueeze(0), eps=1e-8)
print(f"With eps=1e-8: {cos_safe.item()}")


### Source 4: Layer Normalization with Zero Variance

**WHY:** $\text{LN}(x) = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}}$. When all elements equal: $\sigma^2 = 0$, relies entirely on $\epsilon$. With tiny $\epsilon$ (e.g. $10^{-12}$) and **FP16**: $\sqrt{\epsilon}$ may underflow → division by zero.

**Detect:** NaN in first LN layer with constant input; common in FP16 training.

**Fix:** Use $\epsilon \geq 10^{-6}$ for FP16, $\epsilon \geq 10^{-5}$ as safe default.


In [ ]:
# Source 4: LayerNorm zero variance + tiny eps in FP16
x = torch.ones(1, 10) * 5.0  # zero variance

ln_bad = nn.LayerNorm(10, eps=1e-12)
out_bad = ln_bad(x.half())
print(f"LN eps=1e-12, FP16: NaN={torch.isnan(out_bad).any().item()}")

ln_good = nn.LayerNorm(10, eps=1e-5)
out_good = ln_good(x.half())
print(f"LN eps=1e-5, FP16: NaN={torch.isnan(out_good).any().item()}, mean={out_good.mean().item():.4f}")


### Source 5: Cross-Entropy with Out-of-Range Labels

**WHY:** Labels must satisfy $0 \leq y_i < C$. Label $y=12$ with $C=10$ classes → undefined indexing in CUDA kernel → garbage or NaN.

**Detect:** NaN on first batch; check `labels.min()`, `labels.max()` vs `logits.size(-1)`.

**Fix:** Validate labels before loss; fix data pipeline.


In [ ]:
# Source 5: out-of-range labels
logits = torch.randn(4, 10)
labels = torch.tensor([0, 5, 12, -1])  # 12 and -1 invalid!

print(f"Label range: [{labels.min()}, {labels.max()}], num_classes={logits.size(-1)}")
try:
    loss = F.cross_entropy(logits, labels)
    print(f"Loss: {loss.item()}  NaN={torch.isnan(loss).item()}")
except Exception as e:
    print(f"Error: {e}")

# Fix: validate before loss
valid = labels.min() >= 0 and labels.max() < logits.size(-1)
print(f"Labels valid: {valid.item()}")
labels_fixed = labels.clamp(0, logits.size(-1) - 1)
loss_fixed = F.cross_entropy(logits, labels_fixed)
print(f"Fixed loss: {loss_fixed.item():.4f}")


### Source 6: Empty Batch After Filtering → NaN

**WHY:** After filtering bad samples, batch size may become 0. `tensor.mean()` on empty tensor returns NaN (PyTorch convention: undefined reduction).

**Detect:** NaN loss intermittently; check if DataLoader returns empty batches.

**Fix:** Skip empty batches: `if batch.size(0) == 0: continue`


In [ ]:
# Source 6: empty batch
batch = torch.randn(0, 10)
print(f"Empty batch shape: {batch.shape}")
mean_bad = batch.mean()
print(f"mean(empty): {mean_bad.item()}  (NaN={torch.isnan(mean_bad).item()})")

# Fix
if batch.size(0) > 0:
    mean_good = batch.mean()
    print(f"mean(non-empty): {mean_good.item()}")
else:
    print("Skip: empty batch detected before forward pass")


### Source 7: Embedding with Out-of-Vocabulary Index

**WHY:** `nn.Embedding(vocab_size, dim)` expects indices $\in [0, \text{vocab\_size})$. Index $\geq \text{vocab\_size}$ → out-of-bounds memory access → garbage or NaN.

**Detect:** NaN in embedding output; check max token ID vs vocab size.

**Fix:** Clamp indices: `ids.clamp(0, emb.num_embeddings - 1)` or use `<unk>` token.


In [ ]:
# Source 7: OOV embedding index
emb = nn.Embedding(1000, 64)
ids = torch.tensor([500, 1500])  # 1500 >= 1000!
print(f"Max id={ids.max().item()}, vocab_size={emb.num_embeddings}")

try:
    out_bad = emb(ids)
    print(f"OOV embedding output has NaN: {torch.isnan(out_bad).any().item()}")
except IndexError as e:
    print(f"OOV index crash (expected): {e}")

ids_safe = ids.clamp(0, emb.num_embeddings - 1)
out_safe = emb(ids_safe)
print(f"Clamped ids: {ids_safe.tolist()}, NaN={torch.isnan(out_safe).any().item()}")


### Source 8: Division in Custom Loss Functions (Focal Loss)

**WHY:** Focal loss: $(1-p)^\gamma \cdot (-\log p)$. When $p=0$ and bad implementation: $\log(0)=-\infty$, weight $=1$ → $\infty$. When $p=1$ and wrong class: $0 \cdot (-\log 0) = 0 \cdot \infty = \text{NaN}$.

**Detect:** NaN in object detection / imbalanced classification.

**Fix:** Clamp predictions: `pred.clamp(1e-7, 1 - 1e-7)`.


In [ ]:
# Source 8: focal loss NaN
def focal_loss_bad(pred, target, gamma=2):
    ce = -target * torch.log(pred)  # log(0) = -inf when pred=0
    weight = (1 - pred) ** gamma
    return (weight * ce).mean()

def focal_loss_safe(pred, target, gamma=2):
    pred = pred.clamp(1e-7, 1 - 1e-7)
    ce = -target * torch.log(pred)
    weight = (1 - pred) ** gamma
    return (weight * ce).mean()

pred = torch.tensor([0.0, 0.9, 1.0])
target = torch.tensor([1.0, 0.0, 1.0])  # pred=0, target=1 → log(0)

loss_bad = focal_loss_bad(pred, target)
print(f"Bad focal loss: {loss_bad.item()}  (NaN={torch.isnan(loss_bad).item()})")
loss_safe = focal_loss_safe(pred, target)
print(f"Safe focal loss: {loss_safe.item():.4f}")


### Source 9: NaN from Large Weight Initialization

**WHY:** Weights initialized with $\text{std} \gg 1$ → activations grow as $\text{std}^{L}$ through $L$ layers → exceed FP32 max ($\approx 3.4 \times 10^{38}$) → Inf → NaN in first forward pass.

**Detect:** NaN at step 0; check `out.abs().max()` after first forward.

**Fix:** Kaiming/Xavier init; for transformers use $\mathcal{N}(0, 0.02)$ or $\mu$P scaling.


In [ ]:
# Source 9: large weight initialization
model_bad = nn.Linear(1000, 1000)
nn.init.normal_(model_bad.weight, std=10.0)  # way too large
x = torch.randn(32, 1000)
out_bad = model_bad(x)
print(f"Bad init — max activation: {out_bad.abs().max().item():.2e}, NaN={torch.isnan(out_bad).any().item()}")

model_good = nn.Linear(1000, 1000)
nn.init.kaiming_normal_(model_good.weight, mode='fan_in', nonlinearity='relu')
out_good = model_good(x)
print(f"Kaiming init — max activation: {out_good.abs().max().item():.2e}, NaN={torch.isnan(out_good).any().item()}")


### Source 10: `torch.where` NaN Trap

**WHY:** `torch.where(cond, A, B)` computes **BOTH** branches before selecting! So `torch.where(x > 0, 1/x, 0)` still evaluates `1/0 = \infty` even when $x=0$.

**Detect:** Forward looks fine; NaN appears in backward. `detect_anomaly` points to `DivBackward0`.

**Fix:** Clamp denominator before division, or use masked operations.


In [ ]:
# Source 10: torch.where computes BOTH branches
x = torch.tensor([0.0, 1.0, 2.0], requires_grad=True)
result = torch.where(x > 0, 1.0 / x, torch.zeros_like(x))
print(f"Forward result: {result}")  # looks OK: [0, 1, 0.5]
loss = result.sum()
loss.backward()
print(f"Gradient at x=0: {x.grad[0].item()}  (inf={torch.isinf(x.grad[0]).item()})")

# Fix: clamp before division
x2 = torch.tensor([0.0, 1.0, 2.0], requires_grad=True)
safe_x = x2.clamp(min=1e-8)
result_safe = torch.where(x2 > 0, 1.0 / safe_x, torch.zeros_like(x2))
result_safe.sum().backward()
print(f"Safe gradient at x=0: {x2.grad[0].item()}")


### Source 11: `torch.norm` with Zero Vector (Backward)

**WHY:** $\|x\|_2 = \sqrt{\sum x_i^2}$. Backward: $\frac{\partial \|x\|}{\partial x} = \frac{x}{\|x\|}$. When $x = 0$: $\frac{0}{0} = \text{NaN}$.

**Detect:** NaN gradient after `.norm()` or `.normalize()` on zero vector.

**Fix:** Add epsilon inside norm: `torch.sqrt((x**2).sum() + 1e-8)`.


In [ ]:
# Source 11: norm backward on zero vector
x = torch.zeros(5, requires_grad=True)
norm = torch.linalg.norm(x)
print(f"Forward norm: {norm.item()}")
norm.backward()
print(f"Gradient: {x.grad}  (NaN={torch.isnan(x.grad).any().item()})")

x2 = torch.zeros(5, requires_grad=True)
norm_safe = torch.sqrt((x2 ** 2).sum() + 1e-8)
norm_safe.backward()
print(f"Safe gradient: {x2.grad}  (NaN={torch.isnan(x2.grad).any().item()})")


### Source 12: Attention Score NaN with Very Long Sequences (FP16)

**WHY:** For seq_len > 4096 with FP16: $QK^\top / \sqrt{d}$ can exceed 65504 → Inf before softmax → NaN. Softmax of $[\infty, \infty, \ldots]$ = NaN.

**Detect:** NaN only with long context + FP16; check `scores.max()`.

**Fix:** Flash Attention (fused stable kernel), BF16, or sequence parallelism.


In [ ]:
# Source 12: attention overflow in FP16 (simulated)
seq_len, d = 8192, 64
Q = torch.randn(1, seq_len, d, dtype=torch.float16) * 2
K = torch.randn(1, seq_len, d, dtype=torch.float16) * 2
scores = torch.matmul(Q, K.transpose(-1, -2)) / (d ** 0.5)
print(f"Max attention score (FP16): {scores.max().item():.1f}")
print(f"Has Inf: {torch.isinf(scores).any().item()}, count>{65504}: {(scores.abs() > 65504).sum().item()}")

# Fix: BF16
scores_bf16 = torch.matmul(Q.to(torch.bfloat16), K.to(torch.bfloat16).transpose(-1, -2)) / (d ** 0.5)
print(f"Max score (BF16): {scores_bf16.max().item():.1f}, Inf={torch.isinf(scores_bf16).any().item()}")


### Source 13: Gradient Explosion through Residual Chains

**WHY:** Residual: $h_L = h_0 + \sum_{l=1}^{L} F_l(h_{l-1})$. Gradient: $\frac{\partial \mathcal{L}}{\partial h_0} = \frac{\partial \mathcal{L}}{\partial h_L} \left(I + \sum_l \frac{\partial F_l}{\partial h_{l-1}}\right)$. Skip connections help but Jacobians $> 1$ still compound exponentially with depth + high LR.

**Detect:** Grad norm spikes before NaN; monitor per-layer grad norms.

**Fix:** Pre-LN, grad clip (max_norm=1.0), lower LR, weight decay.


In [ ]:
# Source 13: gradient growth through deep residual stack
class ResBlock(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.lin = nn.Linear(d, d)
        nn.init.normal_(self.lin.weight, std=0.5)  # Jacobian can be > 1
    def forward(self, x):
        return x + self.lin(x)

deep = nn.Sequential(*[ResBlock(64) for _ in range(20)])
x = torch.randn(4, 64, requires_grad=True)
out = deep(x)
out.sum().backward()

grad_norms = []
for i, blk in enumerate(deep):
    if blk.lin.weight.grad is not None:
        grad_norms.append(blk.lin.weight.grad.norm().item())

plt.figure(figsize=(10, 4))
plt.plot(grad_norms, 'o-')
plt.xlabel('Layer (0=shallow, 19=deep)'); plt.ylabel('Weight grad norm')
plt.title('Gradient norms through 20-layer residual stack')
plt.tight_layout(); plt.show()
print(f"Shallow grad norm: {grad_norms[0]:.4f}, Deep grad norm: {grad_norms[-1]:.4f}")
print("Fix: Pre-LN, clip_grad_norm_(model.parameters(), 1.0), lower LR")


### Source 14: NaN in Mixed Precision Backward (Underflow)

**WHY:** FP16 smallest subnormal $\approx 5.96 \times 10^{-8}$. Gradients smaller than this **round to zero**. Downstream layer divides by this zero gradient → NaN in second-order effects or custom ops.

**Detect:** Loss scale stuck at 1; tiny gradients in FP16 master weights not updating FP16 cast weights.

**Fix:** BF16 (wider subnormal range), FP32 master weights (AMP default), gradient accumulation in FP32.


In [ ]:
# Source 14: FP16 gradient underflow
grad_fp16 = torch.tensor(1e-9, dtype=torch.float16)
grad_fp32 = torch.tensor(1e-9, dtype=torch.float32)
print(f"1e-9 in FP16: {grad_fp16.item()}  (underflowed to 0)")
print(f"1e-9 in FP32: {grad_fp32.item():.2e}")

# Simulate: division by underflowed grad downstream
tiny = torch.tensor(0.0, dtype=torch.float16)  # underflowed grad stored as 0
# Custom op: scale by inverse grad magnitude
if tiny.item() == 0:
    print("Downstream op sees zero grad → potential div-by-zero in custom normalization")
print("Fix: BF16, accumulate grads in FP32, use GradScaler")


### Source 15: GAN Training NaN (Discriminator Too Strong)

**WHY:** Generator loss: $-\log D(G(z))$. When $D(G(z)) \to 0$ (discriminator perfect): $-\log(0) = \infty$ → NaN. Also: $0 \cdot \log 0$ in some GAN variants.

**Detect:** NaN after discriminator reaches high accuracy; generator loss spikes.

**Fix:** WGAN-GP (Wasserstein + gradient penalty), spectral norm on D, label smoothing, feature matching, two-timescale update (TTUR).


In [ ]:
# Source 15: GAN generator loss NaN when D is too strong
def gan_g_loss(D_fake):
    return -torch.log(D_fake)

D_outputs = torch.tensor([0.5, 0.1, 0.01, 1e-8, 0.0])
for d_val in D_outputs:
    loss = gan_g_loss(d_val)
    print(f"D(G(z))={d_val:.1e} → G_loss={loss.item():.4f}  NaN={torch.isnan(loss).item()}")

print("\nFixes:")
print("  - WGAN-GP: use Wasserstein loss (no log)")
print("  - Label smoothing: D targets in [0.9, 1.0] not 1.0")
print("  - Spectral norm on D: bounds Lipschitz constant")
print("  - Clamp D output: D_fake.clamp(1e-7, 1-1e-7)")


## 1. Why Loss Goes NaN — Root Causes

```
Loss is NaN?
  ├─ Softmax overflow? (x > 88.7 in FP32)
  ├─ Division by zero in norm?
  ├─ Exploding attention logits?
  ├─ LR too high?
  └─ Bad initialization?
```


## 2. Softmax Overflow & Log-Sum-Exp

$e^{88.7} \approx 1.6 \times 10^{38}$ → FP32 overflow.

**Stable softmax:** subtract $c = \max(x)$:

$$
\log\sum_i e^{x_i} = c + \log\sum_i e^{x_i - c}
$$


In [ ]:
def naive_softmax(x):
    exp_x = torch.exp(x)
    return exp_x / exp_x.sum()

def stable_softmax(x):
    c = x.max()
    exp_x = torch.exp(x - c)
    return exp_x / exp_x.sum()

x = torch.tensor([1000.0, 1001.0, 999.0])
print('Naive:', naive_softmax(x))  # nan
print('Stable:', stable_softmax(x))


## 3. Attention Instability

$$
\text{Attn} = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)
$$

If $Q, K$ grow unbounded, logits explode before softmax. **Fix:** Pre-LN, scaled dot-product, lower LR.


## 4. Pre-LN vs Post-LN

**Post-LN:** $x + \text{Attn}(\text{LN}(x))$ — gradients through LN can vanish early in training.

**Pre-LN:** $\text{LN}(x + \text{Attn}(x))$ — better gradient norms at initialization (Xiong et al. 2020).


## 5. Spectral Normalization

$$
\bar{W} = \frac{W}{\sigma(W)}
$$

Constrains Lipschitz constant of linear layer → bounds gradient magnitude.


## 6. Multimodal-Specific: CLIP Temperature Collapse

When learnable $\tau \to 0$, similarities are divided by tiny $\tau$ → logits $\to \pm\infty$.

**Fix:** clamp $\tau \geq \tau_{min}$ (e.g. 0.01) or use fixed $\tau = 0.07$.


In [ ]:
def clip_loss(similarities, tau):
    logits = similarities / tau
    labels = torch.arange(similarities.size(0))
    return F.cross_entropy(logits, labels)

S = torch.randn(4, 4) * 0.5
for tau in [0.07, 0.001, 0.0001]:
    loss = clip_loss(S, tau)
    print(f'tau={tau}: loss={loss.item():.4f}')


In [ ]:
# Gradient monitor — detect instability before NaN
class GradMonitor:
    def __init__(self, threshold=100.0):
        self.threshold = threshold
    def check(self, model):
        total = 0.0
        for p in model.parameters():
            if p.grad is not None:
                total += p.grad.norm().item() ** 2
        total = total ** 0.5
        if total > self.threshold or np.isnan(total):
            print(f'WARNING: grad norm = {total}')
        return total

m = nn.Linear(10, 10)
x = torch.randn(4, 10) * 100  # large input -> potential instability
m(x).sum().backward()
GradMonitor(threshold=50).check(m)


## 7. Modality Collapse

One encoder receives all gradient signal; other modality ignored. **Mitigation:** balanced loss weights, gradient clipping per tower, stop-gradient on dominant branch.


## 8. muP — Width-Independent Hyperparameters

Scale learning rate and initialization with width so optimal $\eta$ does not change when model width increases (Yang et al. 2022). Critical for scaling law experiments.


In [ ]:
# Reproduce softmax overflow then fix with log-sum-exp loss
def log_softmax_stable(x):
    c = x.max(dim=-1, keepdim=True).values
    return x - c - torch.log(torch.exp(x - c).sum(dim=-1, keepdim=True))

logits = torch.tensor([[1000., 1001., 999.]])
print('Stable log-softmax:', log_softmax_stable(logits))


## 9. Instability Fix Decision Table

| Issue | Fix | Priority |
|-------|-----|----------|
| Softmax NaN | Log-sum-exp | Immediate |
| Transformer NaN | Pre-LN | High |
| CLIP NaN | Clamp tau | High |
| Grad spikes | Monitor + clip | Preventive |


## 10. Multi-GPU NaN — Why Single-GPU Fixes Don't Always Work

On one GPU, NaN usually means bad numerics locally. On **multi-GPU**, a NaN on **one rank** poisons **all ranks** via AllReduce:

$$
g = \frac{1}{G}\sum_{r=1}^{G} g_r \quad\Rightarrow\quad \text{if any } g_r = \text{NaN},\; g = \text{NaN}
$$

This section covers the **10 most common multi-GPU NaN root causes** with runnable PyTorch demos (CPU-safe; patterns apply on CUDA + DDP).


## 11. FP16/BF16 Overflow in AMP (Most Common Multi-GPU NaN)

**FP16 max representable** $\approx 65504$. Values above overflow to $\infty$; $\infty - \infty \to$ NaN.

With AMP, loss is scaled before backward:

$$
\text{scaled\_loss} = s \cdot \mathcal{L}, \quad s \text{ starts high (e.g. } 65536\text{)}
$$

If $\mathcal{L} \cdot s > 65504$ in FP16 forward activations → overflow → NaN.

**BF16** shares FP32 exponent range (no 65504 cliff) but lower mantissa precision — preferred on A100/H100 for multi-GPU training.


In [ ]:
# FP16 overflow demo vs BF16 stability
import torch

FP16_MAX = torch.finfo(torch.float16).max
print(f"FP16 max: {FP16_MAX.item():.0f}")

# Simulate CLIP-like logits / tau in FP16
tau = torch.tensor(0.001, dtype=torch.float16)
similarities = torch.tensor([30.0, 25.0, 20.0], dtype=torch.float16)
logits_fp16 = similarities / tau
print(f"logits/tau in FP16: {logits_fp16}  (max={logits_fp16.max().item():.0f})")
print(f"Any Inf? {torch.isinf(logits_fp16).any().item()}")

# Same in BF16 — same exponent range as FP32
logits_bf16 = (similarities.float() / tau.float()).to(torch.bfloat16)
print(f"logits/tau in BF16 max: {logits_bf16.max().item():.0f}, Inf? {torch.isinf(logits_bf16).any().item()}")


## 12. Dynamic Loss Scaling — Math & Failure Mode

PyTorch `GradScaler` adjusts $s$ each step:

- **NaN/Inf in grads** → skip step, $s \leftarrow s / 2$
- **$N$ consecutive good steps** → $s \leftarrow \min(2s, s_{max})$

When $s \to 1$ after many skips, effective update $\approx \eta \cdot g / s$ becomes tiny → **training stops learning** (looks alive, loss flatlines, then eventual NaN from stale numerics).


In [ ]:
# Simulate dynamic loss scale behavior (PyTorch GradScaler logic)
class LossScaleMonitor:
    """Track AMP loss scale over time, alert when dropping."""
    def __init__(self, init_scale=65536.0, growth_factor=2.0, backoff_factor=0.5,
                 growth_interval=2000, alert_threshold=1.0):
        self.scale = init_scale
        self.growth_factor = growth_factor
        self.backoff_factor = backoff_factor
        self.growth_interval = growth_interval
        self.alert_threshold = alert_threshold
        self.history = []
        self.good_steps = 0

    def step(self, had_nan: bool):
        if had_nan:
            self.scale *= self.backoff_factor
            self.good_steps = 0
        else:
            self.good_steps += 1
            if self.good_steps >= self.growth_interval:
                self.scale *= self.growth_factor
                self.good_steps = 0
        self.history.append(self.scale)
        if self.scale <= self.alert_threshold:
            print(f"ALERT: loss scale dropped to {self.scale:.1f} — check LR and data quality")
        return self.scale

monitor = LossScaleMonitor()
# Simulate 5 NaN events then recovery
for step in range(50):
    had_nan = step in {3, 4, 5, 6, 7}
    s = monitor.step(had_nan)
    if step < 12 or step > 45:
        print(f"step {step:2d}: scale={s:8.1f}  nan={had_nan}")

plt.figure(figsize=(10, 4))
plt.plot(monitor.history)
plt.axhline(65504, color='r', ls='--', label='FP16 max (65504)')
plt.xlabel('Step'); plt.ylabel('Loss scale'); plt.title('Dynamic Loss Scale After NaN Skips')
plt.legend(); plt.tight_layout(); plt.show()


## 13. Gradient Accumulation + AMP — Wrong vs Right

When accumulating $K$ micro-batches, **unscale once** after all micro-steps:

**Wrong:** `scaler.unscale_(optimizer)` inside the micro-step loop → divides grads by $s$ multiple times → wrong magnitude → NaN.

**Right:** accumulate scaled grads for $K$ steps, then `scaler.unscale_` once, clip, `scaler.step`.


In [ ]:
# Wrong vs right gradient accumulation + AMP pattern (conceptual, CPU)
import torch
from torch.cuda.amp import GradScaler

def wrong_pattern(model, optimizer, micro_batches, accum_steps=4):
    """ANTI-PATTERN: unscale inside micro-step loop."""
    scaler = GradScaler()
    optimizer.zero_grad()
    for i, batch in enumerate(micro_batches):
        with torch.cuda.amp.autocast(enabled=False):  # CPU demo
            loss = model(batch).sum() / accum_steps
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)  # WRONG: called every micro-step
        if (i + 1) % accum_steps == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

def right_pattern(model, optimizer, micro_batches, accum_steps=4):
    """CORRECT: unscale once after accumulation."""
    scaler = GradScaler()
    optimizer.zero_grad()
    for i, batch in enumerate(micro_batches):
        with torch.cuda.amp.autocast(enabled=False):
            loss = model(batch).sum() / accum_steps
        scaler.scale(loss).backward()
        if (i + 1) % accum_steps == 0:
            scaler.unscale_(optimizer)  # RIGHT: once per optimizer step
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

model = nn.Linear(4, 2)
opt = torch.optim.SGD(model.parameters(), lr=0.1)
batches = [torch.randn(2, 4) for _ in range(8)]
right_pattern(model, opt, batches)
print("Correct pattern: unscale AFTER all micro-steps in accumulation window")


## 14. Cross-GPU Gradient Sync — NaN Propagation

If rank 2 produces NaN gradients, AllReduce average is NaN on **every** rank.

**Fix:** gradient sentinel — detect NaN **before** sync, zero out bad grads on that rank (or skip batch).


In [ ]:
class GradientSentinel:
    """Check gradients before AllReduce, replace NaN/Inf with zero."""
    def __init__(self, rank=0):
        self.rank = rank
        self.nan_events = 0

    def sanitize(self, model) -> bool:
        had_bad = False
        for name, p in model.named_parameters():
            if p.grad is None:
                continue
            if torch.isnan(p.grad).any() or torch.isinf(p.grad).any():
                had_bad = True
                self.nan_events += 1
                print(f"[rank {self.rank}] NaN/Inf in {name} — zeroing grad")
                p.grad = torch.nan_to_num(p.grad, nan=0.0, posinf=0.0, neginf=0.0)
        return had_bad

def simulate_allreduce(grads_per_rank):
    """Simulate mean AllReduce across virtual GPUs."""
    stacked = torch.stack(grads_per_rank)
    if torch.isnan(stacked).any():
        print("AllReduce result: NaN (poisoned by at least one rank)")
    return stacked.mean(dim=0)

# Virtual 4-GPU: rank 2 has NaN gradient
grads = [torch.tensor([1.0, 2.0]), torch.tensor([0.5, 1.5]),
         torch.tensor([float('nan'), 3.0]), torch.tensor([1.2, 0.8])]
print("Before sentinel:", simulate_allreduce(grads))

# Apply sentinel on rank 2
grads[2] = torch.nan_to_num(grads[2], nan=0.0)
print("After sentinel: ", simulate_allreduce(grads))


## 15. Loss Spike → NaN Pattern on Multi-GPU

Typical timeline when one GPU sees a bad batch:

```
Step 1000: loss = 2.3   (normal)
Step 1001: loss = 15.7  (spike — GPU 2 adversarial batch)
Step 1002: loss = 847.2 (exploding — large grad update)
Step 1003: loss = NaN   (dead)
```

After AllReduce, the spike gradient is averaged:

$$
g = \frac{1}{G}\left(g_1 + \cdots + g_{\text{bad}} + \cdots + g_G\right)
$$

Even one large $g_{\text{bad}}$ can dominate if not clipped. **Fix:** per-rank loss clipping, skip batches with loss $> \tau$, monitor grad norm per rank.


In [ ]:
# Simulate loss spike propagation through gradient averaging
G = 4
normal_grads = [torch.randn(100) * 0.01 for _ in range(G)]
bad_grad = torch.randn(100) * 50.0  # spike on rank 2
normal_grads[2] = bad_grad

avg_before_clip = torch.stack(normal_grads).mean(0)
print(f"Global grad norm before clip: {avg_before_clip.norm():.2f}")

# Per-rank clipping before AllReduce
max_norm = 1.0
clipped = []
for g in normal_grads:
    norm = g.norm()
    if norm > max_norm:
        g = g * (max_norm / norm)
    clipped.append(g)
avg_after_clip = torch.stack(clipped).mean(0)
print(f"Global grad norm after per-rank clip: {avg_after_clip.norm():.2f}")

# Per-GPU loss clipping
losses = torch.tensor([2.3, 2.1, 15.7, 2.4])  # rank 2 spike
loss_threshold = 10.0
for r, loss in enumerate(losses):
    if loss > loss_threshold:
        print(f"Rank {r}: skip batch (loss={loss:.1f} > {loss_threshold})")


## 16. BatchNorm NaN in Multi-GPU

Per-GPU batch norm uses **local** batch statistics:

$$
\hat{x} = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}}
$$

When per-GPU batch is tiny ($B_{\text{local}} < 4$), $\sigma^2 \approx 0$ on one GPU → division instability → NaN → sync poisons all ranks.

**Fixes:** `SyncBatchNorm`, or replace with GroupNorm/LayerNorm when $B_{\text{local}}$ is small.


In [ ]:
# BatchNorm NaN with tiny per-GPU batch
def demo_bn_nan(batch_per_gpu=2):
    bn = nn.BatchNorm1d(8)
    bn.train()
    x = torch.ones(batch_per_gpu, 8) * 5.0  # zero variance within batch
    x[:, 0] += torch.randn(batch_per_gpu) * 1e-8
    out = bn(x)
    return torch.isnan(out).any().item(), out

nan_tiny, _ = demo_bn_nan(2)
print(f"BatchNorm with batch=2: NaN={nan_tiny}")

# SyncBatchNorm aggregates stats across GPUs (simulated with larger effective batch)
sync_bn = nn.SyncBatchNorm(8)
sync_bn.train()
# Simulate merged batch from 4 GPUs
merged = torch.randn(8, 8)  # effective batch = 8
out_sync = sync_bn(merged)
print(f"SyncBatchNorm with effective batch=8: NaN={torch.isnan(out_sync).any().item()}")

# GroupNorm fix — no batch-dim dependency
gn = nn.GroupNorm(4, 8)
out_gn = gn(torch.ones(2, 8) * 5.0)
print(f"GroupNorm with batch=2: NaN={torch.isnan(out_gn).any().item()}")


## 17. NCCL Timeout & Hangs (Multi-GPU Failure Mode)

Not always NaN, but the **#2 multi-GPU failure** after numerics:

- Ranks finish at different speeds → NCCL timeout
- **Conditional branching** per rank (`if rank == 0: ...`) without barriers → deadlock
- Debug: `NCCL_DEBUG=INFO`, `TORCH_DISTRIBUTED_DEBUG=DETAIL`

Environment variables:

```
export NCCL_ASYNC_ERROR_HANDLING=1
export TORCH_NCCL_BLOCKING_WAIT=1
export NCCL_DEBUG=INFO
```

**Rule:** every rank must execute the same collective ops in the same order.


## 18. Gradient Norm Explosion on Specific Ranks

Data imbalance → one rank sees harder examples → higher local grad norm:

$$
\lVert g_r \rVert \gg \lVert g_{r'} \rVert \quad \text{for some } r
$$

**Fix:** compute per-rank norm, clip **before** AllReduce, log variance across ranks.


In [ ]:
# Per-rank gradient norm variance simulation
G = 8
torch.manual_seed(0)
grad_norms = []
for rank in range(G):
    # Rank 5 gets harder data (larger gradients)
    scale = 5.0 if rank == 5 else 1.0
    g = torch.randn(1000) * 0.01 * scale
    grad_norms.append(g.norm().item())

print("Per-rank grad norms:", [f"{n:.4f}" for n in grad_norms])
print(f"Mean={np.mean(grad_norms):.4f}, Std={np.std(grad_norms):.4f}, Max/Min={max(grad_norms)/min(grad_norms):.1f}x")

# Clip per rank before sync
max_norm = 1.0
clipped_norms = [min(n, max_norm) for n in grad_norms]
print("After per-rank clip:", [f"{n:.4f}" for n in clipped_norms])


## 19. Mixed Precision Master Weight Desync

AMP keeps **FP32 master weights** $W_{32}$ and casts to FP16 $W_{16}$ for forward.

Smallest FP16 increment: $\Delta_{16} \approx 2^{-24} \approx 5.96 \times 10^{-8}$.

When $\lvert \eta \cdot g \rvert < \Delta_{16}$:

$$
W_{16} = \text{cast}(W_{32}) \quad \text{does not change after update}
$$

Stagnation in FP16 weights → activations drift → eventual overflow/NaN. **Always use FP32 master weights** (PyTorch AMP default with `GradScaler`).


In [ ]:
# FP16 weight stagnation demo
W32 = torch.tensor([1.0], dtype=torch.float32)
lr, grad = 1e-9, 1.0  # tiny update
W32_new = W32 - lr * grad
W16_before = W32.half()
W16_after = W32_new.half()
print(f"FP32 update: {W32.item():.10f} -> {W32_new.item():.10f}")
print(f"FP16 before: {W16_before.item():.6f}, after: {W16_after.item():.6f}, changed: {W16_before.item() != W16_after.item()}")

# Many tiny updates accumulate in FP32 but FP16 appears frozen
W32 = torch.tensor([1.0], dtype=torch.float32)
for _ in range(10000):
    W32 = W32 - 1e-7  # visible in FP32
print(f"After 10k tiny updates: FP32={W32.item():.6f}, FP16={W32.half().item():.6f}")


## 20. Reproducibility Across GPUs (Debugging Confusion)

Non-deterministic ops make multi-GPU bugs **hard to reproduce**:

- `atomicAdd` in CUDA backward
- `cudnn.benchmark = True`
- Different seeds per rank → different data order

```python
torch.use_deterministic_algorithms(True, warn_only=True)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
```

Use **same base seed + rank offset** for DistributedSampler: `seed + rank`.


In [ ]:
# Reproducibility settings (apply before training)
import os

def set_deterministic(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception as e:
        print(f"Deterministic mode note: {e}")

set_deterministic(42)
a = torch.randn(3)
set_deterministic(42)
b = torch.randn(3)
print(f"Same seed -> same tensor: {torch.equal(a, b)}")


## 21. Complete Multi-GPU NaN Debugging Checklist

```
NaN on multi-GPU?
├── Which step? (early vs late)
│   ├── Step 0-10: Initialization problem
│   │   └── Check: init scale, LR warmup
│   ├── Step 10-1000: LR/optimizer issue
│   │   └── Check: LR schedule, loss scale
│   └── Step 1000+: Data/numerical issue
│       └── Check: batch quality, softmax overflow
├── Which rank first?
│   ├── All ranks simultaneously: global issue (LR, loss function)
│   └── Single rank first: data issue on that rank, BatchNorm
├── Using AMP?
│   ├── FP16: likely overflow → try BF16
│   └── BF16: rare overflow → check loss function
├── Loss scale dropping to 1?
│   └── Too many NaN skips → lower LR, check data
└── Grad norm before NaN?
    └── If spike: clip grads, check data quality
```


## 22. NaN Prevention Toolkit

Complete utilities for production multi-GPU training: layer hooks, gradient sentinel, loss scale monitor, rank-aware logging, and a safe training loop skeleton.


In [ ]:
class NaNDetector:
    """Hook that monitors every layer for NaN/Inf in forward/backward."""
    def __init__(self, model, rank=0):
        self.rank = rank
        self.events = []
        self._handles = []
        for name, module in model.named_modules():
            if name == '':
                continue
            self._handles.append(module.register_forward_hook(self._fwd_hook(name)))
            self._handles.append(module.register_full_backward_hook(self._bwd_hook(name)))

    def _fwd_hook(self, name):
        def hook(mod, inp, out):
            t = out if isinstance(out, torch.Tensor) else None
            if t is not None and (torch.isnan(t).any() or torch.isinf(t).any()):
                self.events.append(('forward', name))
                print(f"[rank {self.rank}] FWD NaN/Inf in {name}")
        return hook

    def _bwd_hook(self, name):
        def hook(mod, grad_in, grad_out):
            for g in grad_out:
                if g is not None and (torch.isnan(g).any() or torch.isinf(g).any()):
                    self.events.append(('backward', name))
                    print(f"[rank {self.rank}] BWD NaN/Inf in {name}")
        return hook

    def remove(self):
        for h in self._handles:
            h.remove()


class RankAwareLogger:
    """Per-GPU logging for distributed debugging."""
    def __init__(self, rank=0, world_size=1):
        self.rank = rank
        self.world_size = world_size

    def log(self, msg, main_only=False):
        if main_only and self.rank != 0:
            return
        print(f"[rank {self.rank}/{self.world_size}] {msg}")

    def log_all_ranks(self, values: dict):
        if self.rank == 0:
            print(" | ".join(f"r{k}={v:.4f}" for k, v in sorted(values.items())))


class SafeTrainingLoop:
    """Training loop skeleton with NaN protections (single-process demo)."""
    def __init__(self, model, optimizer, rank=0, world_size=1, max_grad_norm=1.0):
        self.model = model
        self.optimizer = optimizer
        self.rank = rank
        self.world_size = world_size
        self.max_grad_norm = max_grad_norm
        self.sentinel = GradientSentinel(rank)
        self.scale_monitor = LossScaleMonitor()
        self.logger = RankAwareLogger(rank, world_size)
        self.detector = NaNDetector(model, rank)

    def train_step(self, loss: torch.Tensor) -> bool:
        if torch.isnan(loss) or torch.isinf(loss):
            self.logger.log(f"Bad loss={loss.item()}, skipping step")
            self.scale_monitor.step(had_nan=True)
            return False
        self.optimizer.zero_grad()
        loss.backward()
        self.sentinel.sanitize(self.model)
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.max_grad_norm)
        self.optimizer.step()
        self.scale_monitor.step(had_nan=False)
        return True

# Quick demo
m = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 2))
opt = torch.optim.Adam(m.parameters(), lr=1e-3)
loop = SafeTrainingLoop(m, opt)
x = torch.randn(4, 10)
ok = loop.train_step(m(x).sum())
print(f"Step OK: {ok}, loss scale: {loop.scale_monitor.scale}")
loop.detector.remove()


## 23. Scenario 1 — CLIP NaN at Step 500 on 4 GPUs

**Symptoms:** NaN around step 500, 4× GPU DDP, FP16, learnable temperature, no grad clip.

**Diagnosis:** $\tau \to 0.001$ → logits $s/\tau > 65504$ in FP16.

**Fix:** clamp $\tau \geq 0.01$, switch to BF16, add `clip_grad_norm_(..., 1.0)`.


In [ ]:
# Scenario 1: CLIP temperature collapse in FP16
class LearnableTemperatureCLIP(nn.Module):
    def __init__(self, dim=4, tau_init=0.07, tau_min=0.01):
        super().__init__()
        self.log_tau = nn.Parameter(torch.tensor(np.log(tau_init)))
        self.tau_min = tau_min

    def forward(self, sim):
        tau = self.log_tau.exp().clamp(min=self.tau_min)
        logits = sim / tau
        labels = torch.arange(sim.size(0), device=sim.device)
        return F.cross_entropy(logits, labels), tau

# Step 1: show FP16 overflow when tau is tiny
S_fp16 = torch.tensor([[30., 25., 20., 15.]], dtype=torch.float16)
for tau_val in [0.07, 0.01, 0.001]:
    logits = S_fp16 / tau_val
    overflow = logits.max().item() > 65504 or torch.isinf(logits).any().item()
    print(f'tau={tau_val}: max_logit={logits.max().item():.0f}, FP16_overflow={overflow}')

# Step 2: FP16 overflow logits -> NaN loss (the actual step-500 failure mode)
overflow_logits = torch.tensor([[70000., 69000., 68000., 67000.]], dtype=torch.float16)
bad_loss = F.cross_entropy(overflow_logits, torch.tensor([0]))
print(f'FP16 overflow logits -> loss={bad_loss.item()} (NaN={torch.isnan(bad_loss).item()})')

healthy_logits = (S_fp16 / 0.01).squeeze(0)
healthy_loss = F.cross_entropy(healthy_logits.unsqueeze(0), torch.tensor([0]))
print(f'Clamped tau=0.01 loss={healthy_loss.item():.4f} (healthy)')

def train_clip_scenario(use_clamp=True, aggressive_lr=0.15):
    torch.manual_seed(42)
    model = LearnableTemperatureCLIP(tau_min=0.01 if use_clamp else 1e-6)
    opt = torch.optim.Adam([model.log_tau], lr=aggressive_lr)
    S = torch.randn(4, 4) * 2.0  # larger similarities -> faster collapse
    nan_step = None
    for step in range(300):
        opt.zero_grad()
        loss, tau = model(S)
        if torch.isnan(loss) or torch.isinf(loss):
            nan_step = step
            break
        loss.backward()
        torch.nn.utils.clip_grad_norm_([model.log_tau], 10.0 if use_clamp else float('inf'))
        opt.step()
    return nan_step, model.log_tau.exp().item()

nan_no_clamp, tau_bad = train_clip_scenario(use_clamp=False)
nan_clamp, tau_good = train_clip_scenario(use_clamp=True)
print(f'Without clamp: NaN at step {nan_no_clamp}, final tau={tau_bad:.6f}')
print(f'With clamp:    NaN at step {nan_clamp}, final tau={tau_good:.6f}')
print('Fix: clamp tau>=0.01, use BF16, clip_grad_norm=1.0 on all params')


## 24. Scenario 2 — Fine for 10k Steps, Then Sudden NaN on 8 GPUs

**Symptoms:** AMP FP16, loss scale stuck at 1, grad norms creeping up.

**Root cause:** weights grow → activations grow → FP16 overflow.

**Fix:** weight decay, grad clip, switch to BF16, monitor loss scale.


In [ ]:
# Scenario 2: monitor loss scale + grad norms over training
class GrowingModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.w = nn.Parameter(torch.ones(1) * 0.1)

    def forward(self, x):
        return (self.w * x).pow(2).mean()

model = GrowingModel()
opt = torch.optim.SGD(model.parameters(), lr=0.5, weight_decay=0.0)  # no WD -> grows
monitor = LossScaleMonitor(init_scale=65536.0)
grad_norms, scales = [], []

for step in range(300):
    x = torch.randn(32)
    opt.zero_grad()
    loss = model(x)
    loss.backward()
    gn = torch.nn.utils.clip_grad_norm_([model.w], float('inf')).item()
    grad_norms.append(gn)
    opt.step()
    had_nan = np.isnan(gn) or gn > 1e4
    scales.append(monitor.step(had_nan))
    if had_nan:
        print(f"Step {step}: grad_norm={gn:.2e}, w={model.w.item():.4f}")
        break

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(grad_norms); ax[0].set_title('Grad norm creep (no weight decay)'); ax[0].set_xlabel('Step')
ax[1].plot(scales); ax[1].set_title('Loss scale'); ax[1].set_xlabel('Step')
plt.tight_layout(); plt.show()
print("Fix: add weight_decay=0.01, clip_grad_norm=1.0, use bfloat16")


## 25. Scenario 3 — NaN Only on Rank 3 (Then All Ranks)

**Symptoms:** rank 3 logs NaN first; shortly after all ranks NaN.

**Root cause:** corrupted input on rank 3 → NaN embedding → AllReduce poisons everyone.

**Fix:** validate data, check for NaN **before** backward, skip bad batches.


In [ ]:
class DataValidator:
    """Validate batches before forward; skip corrupted samples."""
    def __init__(self, rank=0):
        self.rank = rank
        self.skipped = 0

    def validate(self, batch: torch.Tensor) -> bool:
        if torch.isnan(batch).any() or torch.isinf(batch).any():
            self.skipped += 1
            print(f"[rank {self.rank}] SKIP corrupted batch (total skipped={self.skipped})")
            return False
        if batch.abs().max() > 1e4:
            self.skipped += 1
            print(f"[rank {self.rank}] SKIP extreme values max={batch.abs().max():.2e}")
            return False
        return True

def embed(x):
    return x @ torch.tensor([[1.0, float('nan')], [0.5, 0.5]])  # corrupted path

validator = DataValidator(rank=3)
batches = [torch.randn(4, 2) for _ in range(5)]
batches[2][0, 1] = float('nan')  # corrupted batch on rank 3

for i, batch in enumerate(batches):
    if not validator.validate(batch):
        continue
    out = embed(batch)
    if torch.isnan(out).any():
        print(f"Would poison AllReduce at batch {i} without validator")
    else:
        print(f"Batch {i} OK")


## 26. Scenario 4 — LLaVA Finetune NaN with LoRA + Multi-GPU

**Symptoms:** BF16 + LoRA rank 16 + FSDP, NaN in vision projector.

**Root cause:** LoRA scaling $\alpha/r$ too high → large delta weights → overflow in projector.

**Fix:** lower $\alpha$, BF16 (not FP16), `clip_grad_norm=1.0`, sane FSDP mixed precision.


In [ ]:
# Scenario 4: LoRA scaling and safe config
class LoRALinear(nn.Module):
    def __init__(self, in_f, out_f, rank=16, alpha=16):
        super().__init__()
        self.base = nn.Linear(in_f, out_f, bias=False)
        self.base.weight.requires_grad_(False)
        self.A = nn.Parameter(torch.randn(rank, in_f) * 0.01)
        self.B = nn.Parameter(torch.zeros(out_f, rank))
        self.scaling = alpha / rank

    def forward(self, x):
        return self.base(x) + (x @ self.A.T @ self.B.T) * self.scaling

def lora_forward_norm(alpha, rank=16):
    layer = LoRALinear(512, 512, rank=rank, alpha=alpha)
    x = torch.randn(8, 512)
    out = layer(x)
    return out.abs().max().item()

for alpha in [16, 64, 256]:
    mx = lora_forward_norm(alpha)
    print(f"alpha={alpha:3d}, alpha/r={alpha/16:.1f}, max activation={mx:.2f}")

print('Safe LLaVA + LoRA + FSDP config:')
print('  - alpha=16, rank=16 (scaling=1.0)')
print('  - dtype=bfloat16')
print('  - clip_grad_norm=1.0')
print('  - FSDP MixedPrecision(param_dtype=bfloat16, reduce_dtype=float32)')


## 27. Updated Instability Fix Table (Multi-GPU)

| Issue | Fix | Priority |
|-------|-----|----------|
| FP16 overflow | BF16 + loss scaling | Immediate |
| NaN on one rank | Gradient sentinel + data validation | Immediate |
| Loss scale → 1 | Lower LR, fix data, fewer NaN batches | High |
| BatchNorm NaN | SyncBatchNorm or GroupNorm | High |
| Grad accum + AMP | Unscale once after accumulation | High |
| CLIP NaN | Clamp τ, BF16, grad clip | High |
| NCCL hang | Same collectives all ranks; NCCL_DEBUG | Medium |


## 28. Systematic NaN Debugging Methodology

When loss goes NaN, follow this **8-step procedure** (don't guess randomly):

```
1. REPRODUCE  — Find minimum steps to NaN (reduce data, smaller model, single GPU)
2. LOCATE     — torch.autograd.detect_anomaly() → which backward op
3. TRACE      — Register forward hooks on every layer → first NaN layer
4. ISOLATE    — Forward NaN or backward NaN?
                  Forward  → bad data or numerical overflow in computation
                  Backward → gradient explosion or instability in loss function
5. IDENTIFY   — Match against the 15 NaN sources in Section 0d above
6. FIX        — Apply the specific fix for that source
7. VERIFY     — Run for 2× the original NaN step count without NaN
8. HARDEN     — Add permanent NaN hooks / sentinel for production
```

### Decision Tree

```
NaN detected
├── Step 0?        → Check initialization (Source 9)
├── Step 1-100?    → Check LR, warmup, labels (Sources 5, 6)
├── During loss?   → Check log/softmax/KL (Sources 1, 2, 8, 15)
├── In norm layer? → Check LN/BN eps, zero variance (Source 4)
├── In attention?  → Check FP16 overflow, long seq (Source 12)
├── In backward?   → detect_anomaly + NaNTracer (Sources 10, 11)
└── Multi-GPU?     → Check AllReduce propagation (Section 10+)
```


## 29. NaNTracer — Complete Layer-by-Layer Utility

Attach `NaNTracer` to any model to find the **exact layer and direction** (forward vs backward) of the first NaN/Inf.


In [ ]:
class NaNTracer:
    """Attach to model; traces exact layer and direction of first NaN."""
    def __init__(self, model):
        self.model = model
        self.first_nan = None
        self._hooks = []
        for name, mod in model.named_modules():
            if name == '':
                continue
            self._hooks.append(mod.register_forward_hook(self._make_fwd(name)))
            self._hooks.append(mod.register_full_backward_hook(self._make_bwd(name)))

    def _make_fwd(self, name):
        def hook(mod, inp, out):
            if self.first_nan:
                return
            outs = [out] if isinstance(out, torch.Tensor) else (
                list(out) if isinstance(out, tuple) else [])
            for o in outs:
                if isinstance(o, torch.Tensor) and (torch.isnan(o).any() or torch.isinf(o).any()):
                    n_nan = torch.isnan(o).sum().item()
                    n_inf = torch.isinf(o).sum().item()
                    self.first_nan = f"FORWARD in '{name}': {n_nan} NaN, {n_inf} Inf"
                    print(f"*** FIRST NaN: {self.first_nan}")
                    for i, inp_t in enumerate(inp):
                        if isinstance(inp_t, torch.Tensor):
                            print(f"    Input[{i}]: shape={inp_t.shape}, "
                                  f"has_nan={torch.isnan(inp_t).any().item()}, "
                                  f"range=[{inp_t.min().item():.4f}, {inp_t.max().item():.4f}]")
        return hook

    def _make_bwd(self, name):
        def hook(mod, grad_in, grad_out):
            if self.first_nan:
                return
            for i, g in enumerate(grad_out):
                if g is not None and (torch.isnan(g).any() or torch.isinf(g).any()):
                    self.first_nan = f"BACKWARD in '{name}': grad_out[{i}] has NaN/Inf"
                    print(f"*** FIRST NaN: {self.first_nan}")
        return hook

    def remove(self):
        for h in self._hooks:
            h.remove()

    def report(self):
        if self.first_nan:
            print(f"\nNaN Source: {self.first_nan}")
        else:
            print("\nNo NaN detected in this forward+backward pass")
        return self.first_nan


# Demo: trace NaN through a small network
class BadNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(4, 8)
        self.fc2 = nn.Linear(8, 4)
        self.fc2.weight.data.zero_()
        self.fc2.bias.data.zero_()
    def forward(self, x):
        h = F.relu(self.fc1(x))
        # Zero fc2 weights → zero output → div by zero norm
        return self.fc2(h) / h.norm(dim=-1, keepdim=True)

model = BadNet()
tracer = NaNTracer(model)
x = torch.randn(2, 4)
try:
    with torch.autograd.detect_anomaly():
        out = model(x)
        out.sum().backward()
except RuntimeError:
    pass
tracer.report()
tracer.remove()


## 30. NaN Prevention Recipes — Model-Type Summary

| Model Type | Common NaN Source | Prevention Recipe |
|------------|-------------------|-------------------|
| CLIP / Contrastive | τ collapse, FP16 logit overflow | Clamp τ≥0.01, BF16, grad clip 1.0 |
| Transformer LM | Attention overflow, Post-LN instability | Pre-LN, flash attention, BF16 |
| Image Captioning | log(0) in CE, empty caption | Label smoothing, min caption len=1 |
| VQA | Soft CE with zero probs | Clamp probs, use log_softmax |
| LoRA Finetuning | Large α/r scaling | α=r, grad clip 1.0 |
| Diffusion | Noise schedule edge cases | Clamp σ, stable loss |
| GAN | −log(D(G(z))) with D→0 | WGAN-GP, spectral norm |
| Multi-GPU DDP | AllReduce NaN propagation | GradientSentinel, data validation |
| FSDP | Mixed precision casting | bf16/fp32/fp32 combination |
| DeepSpeed ZeRO | Sharded grad NaN localization | safe_get_full_grad, grad clipping |

**Golden rules:**
1. Always use `detect_anomaly()` first when debugging
2. Prefer BF16 over FP16 on Ampere+ GPUs
3. Always `clip_grad_norm_(..., 1.0)` for transformers
4. Validate data before forward (no NaN inputs, valid labels)
5. Monitor loss scale and grad norms proactively


## References & Further Reading

### Papers
- Micikevicius et al. (2018) — Mixed Precision Training — [arXiv:1710.03740](https://arxiv.org/abs/1710.03740)
- Ott et al. (2019) — fairseq: FP16 Training — [arXiv:1904.10509](https://arxiv.org/abs/1904.10509)
- Xiong et al. (2020) — On Layer Normalization in Transformers — [arXiv:2002.04745](https://arxiv.org/abs/2002.04745)
- Yang et al. (2022) — Tensor Programs V (μP) — [arXiv:2203.03466](https://arxiv.org/abs/2203.03466)
- Miyato et al. (2018) — Spectral Normalization — [arXiv:1802.05957](https://arxiv.org/abs/1802.05957)
- Goyal et al. (2017) — Accurate, Large Minibatch SGD — [arXiv:1706.02677](https://arxiv.org/abs/1706.02677)

### Blogs & Docs
- [Lilian Weng — Training Large Neural Networks](https://lilianweng.github.io/posts/2021-09-25-train-compute/)
- [HuggingFace — Debugging Mixed Precision](https://huggingface.co/docs/transformers/perf_train_gpu_one#mixed-precision-training)
- [PyTorch AMP docs](https://pytorch.org/docs/stable/amp.html)
- [NVIDIA — Mixed Precision Training](https://docs.nvidia.com/deeplearning/performance/mixed-precision-training/)
